# SQL Analytics — E-Commerce Business Intelligence
### Portfolio Project | Data Analyst | Istanbul Job Market

---

## What this project demonstrates

This project shows end-to-end SQL analytics on a realistic Turkish e-commerce dataset — from raw tables to business insights — using only SQL queries run inside Python via SQLite.


## Business scenario

We are analysts at a Turkish e-commerce platform. The business wants answers to seven key questions:

| # | Business Question | SQL Concept |
|---|---|---|
| 1 | What are our monthly revenue trends? | GROUP BY, date functions |
| 2 | Which product categories drive the most revenue? | Aggregations, ORDER BY |
| 3 | Who are our top customers by lifetime value? | Subqueries, ranking |
| 4 | How is revenue growing month over month? | Window functions — LAG |
| 5 | What does the sales funnel look like? | CTEs, multi-step logic |
| 6 | Which customers are at risk of churning? | Window functions — DATEDIFF |
| 7 | What is the repeat purchase rate by cohort? | CTEs, self-joins |

---
**Tools:** Python · SQLite · Pandas · Plotly · Matplotlib

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'grid.color': '#21262d',
    'axes.labelcolor': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'text.color': '#c9d1d9', 'font.size': 11
})

SEED = 42
np.random.seed(SEED)
print('Ready.')

## Step 1 — Database Schema & Data

We create four tables that mirror a real e-commerce data warehouse:

```
customers (customer_id, name, city, signup_date, segment)
    │
    └── orders (order_id, customer_id, order_date, status, city)
            │
            └── order_items (item_id, order_id, product_id, quantity, unit_price)
                    │
                    └── products (product_id, name, category, cost_price)
```

This is a standard **star schema** — the most common structure in real analytics databases. `orders` is the fact table; `customers` and `products` are dimension tables.

### Why SQLite?
SQLite runs entirely in memory — no server setup needed. The SQL syntax is 95% identical to PostgreSQL and MySQL, which are used in production. Every query here would run on those databases with minimal changes.

In [ ]:
# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

# ── CREATE TABLES ─────────────────────────────────────────────
cur.executescript("""
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    name          TEXT,
    city          TEXT,
    signup_date   DATE,
    segment       TEXT   -- Premium, Standard, Budget
);

CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    name          TEXT,
    category      TEXT,
    cost_price    REAL
);

CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER REFERENCES customers(customer_id),
    order_date    DATE,
    status        TEXT   -- completed, returned, cancelled
);

CREATE TABLE order_items (
    item_id       INTEGER PRIMARY KEY,
    order_id      INTEGER REFERENCES orders(order_id),
    product_id    INTEGER REFERENCES products(product_id),
    quantity      INTEGER,
    unit_price    REAL
);
""")

# ── POPULATE WITH REALISTIC DATA ──────────────────────────────
import random
from datetime import date, timedelta
random.seed(SEED)

CITIES    = ['İstanbul','Ankara','İzmir','Bursa','Antalya','Adana','Konya']
CITY_W    = [0.40, 0.18, 0.12, 0.08, 0.07, 0.08, 0.07]
SEGMENTS  = ['Premium','Standard','Budget']
SEG_W     = [0.20, 0.50, 0.30]
CATEGORIES = ['Electronics','Fashion','Home & Kitchen','Beauty','Sportswear','Books']
CAT_PRICE  = [2000, 350, 700, 180, 550, 90]
CAT_COST   = [1640, 193, 490, 81,  330, 58]

START = date(2022, 1, 1)
END   = date(2024, 3, 31)
DAYS  = (END - START).days

# Products
products_data = []
pid = 1
for cat, price, cost in zip(CATEGORIES, CAT_PRICE, CAT_COST):
    for i in range(1, 11):  # 10 products per category
        products_data.append((pid, f'{cat} Product {i}', cat,
                               round(cost * random.uniform(0.85, 1.15), 2)))
        pid += 1
cur.executemany('INSERT INTO products VALUES (?,?,?,?)', products_data)

# Customers
N_CUSTOMERS = 3000
customers_data = []
for cid in range(1, N_CUSTOMERS + 1):
    signup = START + timedelta(days=random.randint(0, DAYS - 30))
    customers_data.append((
        cid, f'Customer_{cid}',
        random.choices(CITIES, CITY_W)[0],
        signup.isoformat(),
        random.choices(SEGMENTS, SEG_W)[0]
    ))
cur.executemany('INSERT INTO customers VALUES (?,?,?,?,?)', customers_data)

# Orders and items
N_ORDERS = 25000
orders_data, items_data = [], []
oid, iid = 1, 1

for _ in range(N_ORDERS):
    cid        = random.randint(1, N_CUSTOMERS)
    doy        = random.randint(0, DAYS)
    order_date = START + timedelta(days=doy)
    # Seasonal boost: Nov peak, summer dip
    month      = order_date.month
    seasonal   = 1.3 if month == 11 else (0.8 if month in [7, 8] else 1.0)
    if random.random() > seasonal * 0.7:
        continue
    status = random.choices(['completed','returned','cancelled'], [0.87, 0.08, 0.05])[0]
    orders_data.append((oid, cid, order_date.isoformat(), status))

    # 1-4 items per order
    n_items = random.choices([1,2,3,4], [0.55,0.25,0.12,0.08])[0]
    for _ in range(n_items):
        prod    = random.choice(products_data)
        cat_idx = CATEGORIES.index(prod[2])
        price   = CAT_PRICE[cat_idx] * random.uniform(0.85, 1.20)
        qty     = random.choices([1,2,3], [0.75, 0.18, 0.07])[0]
        items_data.append((iid, oid, prod[0], qty, round(price, 2)))
        iid += 1
    oid += 1

cur.executemany('INSERT INTO orders VALUES (?,?,?,?)', orders_data)
cur.executemany('INSERT INTO order_items VALUES (?,?,?,?,?)', items_data)
conn.commit()

print('Database ready.')
print(f'  customers:   {cur.execute("SELECT COUNT(*) FROM customers").fetchone()[0]:,}')
print(f'  products:    {cur.execute("SELECT COUNT(*) FROM products").fetchone()[0]:,}')
print(f'  orders:      {cur.execute("SELECT COUNT(*) FROM orders").fetchone()[0]:,}')
print(f'  order_items: {cur.execute("SELECT COUNT(*) FROM order_items").fetchone()[0]:,}')

## Query 1 — Monthly Revenue Trend

### Business question
How is our revenue trending month by month? Are we growing?

### SQL concepts used
- `strftime()` to extract year and month from a date
- `JOIN` to connect orders with their items
- `WHERE status = 'completed'` to exclude returns and cancellations
- `GROUP BY` to aggregate by month
- `ORDER BY` to sort chronologically

### Why filter for completed only?
Returned and cancelled orders should not count as revenue. A common mistake is summing all order values without filtering by status — this overstates revenue.

In [ ]:
q1 = """
SELECT
    strftime('%Y-%m', o.order_date)          AS year_month,
    COUNT(DISTINCT o.order_id)               AS total_orders,
    COUNT(DISTINCT o.customer_id)            AS unique_customers,
    ROUND(SUM(oi.quantity * oi.unit_price), 0) AS total_revenue_try,
    ROUND(AVG(oi.quantity * oi.unit_price), 0) AS avg_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'completed'
GROUP BY strftime('%Y-%m', o.order_date)
ORDER BY year_month
"""

monthly = pd.read_sql(q1, conn)
print('Monthly Revenue (last 6 months):')
print(monthly.tail(6).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Query 1 — Monthly Revenue Trend', fontsize=13, fontweight='bold')

axes[0].plot(monthly.year_month, monthly.total_revenue_try / 1e6,
              color='#f5c842', lw=2.5, marker='o', ms=5)
axes[0].fill_between(monthly.year_month, monthly.total_revenue_try / 1e6,
                      alpha=0.15, color='#f5c842')
axes[0].set_title('Monthly Revenue (M TRY)', fontweight='bold')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('Revenue (M TRY)')
axes[0].tick_params(axis='x', rotation=45); axes[0].grid(True, alpha=0.3)

axes[1].bar(monthly.year_month, monthly.unique_customers,
             color='#5b8ef0', alpha=0.85)
axes[1].set_title('Monthly Unique Customers', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Customers')
axes[1].tick_params(axis='x', rotation=45); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('q1_monthly_revenue.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Query 2 — Revenue by Product Category

### Business question
Which categories generate the most revenue and profit? Where should we focus our inventory investment?

### SQL concepts used
- Three-table JOIN (orders → order_items → products)
- Computed columns: revenue, profit, margin
- `ROUND()` for clean output
- `ORDER BY revenue DESC` to rank by importance

In [ ]:
q2 = """
SELECT
    p.category,
    COUNT(DISTINCT o.order_id)                        AS total_orders,
    SUM(oi.quantity)                                  AS units_sold,
    ROUND(SUM(oi.quantity * oi.unit_price), 0)        AS revenue_try,
    ROUND(SUM(oi.quantity * (oi.unit_price - p.cost_price)), 0) AS profit_try,
    ROUND(
        100.0 * SUM(oi.quantity * (oi.unit_price - p.cost_price))
        / SUM(oi.quantity * oi.unit_price), 1
    )                                                 AS margin_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p     ON oi.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY p.category
ORDER BY revenue_try DESC
"""

cat_df = pd.read_sql(q2, conn)
print('Revenue by Category:')
print(cat_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Query 2 — Revenue & Margin by Category', fontsize=13, fontweight='bold')

colors = ['#f5c842','#5b8ef0','#3de8a0','#f06090','#b07af5','#ff8c42']
axes[0].barh(cat_df.category, cat_df.revenue_try / 1e6,
              color=colors, alpha=0.85)
axes[0].set_xlabel('Revenue (M TRY)')
axes[0].set_title('Revenue by Category', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(cat_df.revenue_try / 1e6):
    axes[0].text(v + 0.05, i, f'{v:.1f}M', va='center', fontsize=10)

axes[1].barh(cat_df.category, cat_df.margin_pct,
              color=colors, alpha=0.85)
axes[1].set_xlabel('Gross Margin (%)')
axes[1].set_title('Profit Margin by Category', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(cat_df.margin_pct):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('q2_category_revenue.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Query 3 — Top Customers by Lifetime Value

### Business question
Who are our most valuable customers? Which city and segment do they come from?

### SQL concepts used
- JOIN across three tables
- `ROW_NUMBER() OVER (ORDER BY ...)` — window function for ranking
- CTE (Common Table Expression) to make the query readable
- `HAVING` to filter after aggregation

### Why use a CTE here?
Without a CTE, we would need a nested subquery which is harder to read and debug. A CTE named `customer_ltv` makes the logic clear: first calculate lifetime value, then rank. This is how professional analysts write SQL.

In [ ]:
q3 = """
WITH customer_ltv AS (
    -- Step 1: Calculate lifetime value per customer
    SELECT
        o.customer_id,
        c.city,
        c.segment,
        COUNT(DISTINCT o.order_id)                      AS total_orders,
        ROUND(SUM(oi.quantity * oi.unit_price), 0)      AS lifetime_value_try,
        ROUND(AVG(oi.quantity * oi.unit_price), 0)      AS avg_order_value,
        MIN(o.order_date)                               AS first_order,
        MAX(o.order_date)                               AS last_order
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c    ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id, c.city, c.segment
    HAVING COUNT(DISTINCT o.order_id) >= 2  -- at least 2 orders (repeat buyers)
)
-- Step 2: Rank by lifetime value and show top 15
SELECT
    ROW_NUMBER() OVER (ORDER BY lifetime_value_try DESC) AS rank,
    customer_id,
    city,
    segment,
    total_orders,
    lifetime_value_try,
    avg_order_value,
    first_order,
    last_order
FROM customer_ltv
ORDER BY lifetime_value_try DESC
LIMIT 15
"""

top_customers = pd.read_sql(q3, conn)
print('Top 15 Customers by Lifetime Value:')
print(top_customers[['rank','customer_id','city','segment','total_orders',
                       'lifetime_value_try','avg_order_value']].to_string(index=False))

# LTV distribution by segment
q3b = """
WITH customer_ltv AS (
    SELECT c.segment, c.city,
        ROUND(SUM(oi.quantity * oi.unit_price), 0) AS ltv
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c    ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id, c.segment, c.city
)
SELECT segment, city,
    COUNT(*)           AS n_customers,
    ROUND(AVG(ltv), 0) AS avg_ltv,
    ROUND(SUM(ltv), 0) AS total_ltv
FROM customer_ltv
GROUP BY segment, city
ORDER BY total_ltv DESC
LIMIT 20
"""
ltv_seg = pd.read_sql(q3b, conn)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Query 3 — Customer Lifetime Value Analysis', fontsize=13, fontweight='bold')

seg_avg = ltv_seg.groupby('segment')['avg_ltv'].mean().sort_values(ascending=False)
seg_colors = ['#f5c842','#5b8ef0','#3de8a0']
axes[0].bar(seg_avg.index, seg_avg.values, color=seg_colors, alpha=0.85, width=0.5)
for i, v in enumerate(seg_avg.values):
    axes[0].text(i, v + 20, f'{v:,.0f} TRY', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Average LTV (TRY)')
axes[0].set_title('Average Customer LTV by Segment', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

city_rev = ltv_seg.groupby('city')['total_ltv'].sum().sort_values(ascending=True)
axes[1].barh(city_rev.index, city_rev.values / 1e6, color='#5b8ef0', alpha=0.85)
axes[1].set_xlabel('Total LTV (M TRY)')
axes[1].set_title('Total Customer LTV by City', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('q3_customer_ltv.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Query 4 — Month-over-Month Revenue Growth

### Business question
How much did revenue grow compared to the previous month? Which months had the biggest jumps?

### SQL concepts used
- `LAG()` window function — looks at the previous row's value
- Arithmetic on window function output to compute growth rate
- CTE to separate the aggregation from the growth calculation

### What is LAG()?
`LAG(revenue, 1) OVER (ORDER BY month)` returns the revenue from the previous month for each row. This lets us compute month-over-month growth without a self-join. It is one of the most useful window functions in analytics.

In [ ]:
q4 = """
WITH monthly_rev AS (
    SELECT
        strftime('%Y-%m', o.order_date)            AS month,
        ROUND(SUM(oi.quantity * oi.unit_price), 0) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY strftime('%Y-%m', o.order_date)
)
SELECT
    month,
    revenue,
    LAG(revenue, 1) OVER (ORDER BY month)  AS prev_month_revenue,
    ROUND(
        100.0 * (revenue - LAG(revenue, 1) OVER (ORDER BY month))
        / LAG(revenue, 1) OVER (ORDER BY month),
    1)                                     AS mom_growth_pct
FROM monthly_rev
ORDER BY month
"""

mom = pd.read_sql(q4, conn).dropna()
print('Month-over-Month Revenue Growth (last 6 months):')
print(mom.tail(6).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Query 4 — Month-over-Month Growth (LAG Window Function)', fontsize=13, fontweight='bold')

axes[0].bar(mom.month, mom.mom_growth_pct,
             color=['#3de8a0' if v >= 0 else '#f06090' for v in mom.mom_growth_pct],
             alpha=0.85)
axes[0].axhline(0, color='white', lw=0.8, alpha=0.4)
axes[0].set_xlabel('Month'); axes[0].set_ylabel('MoM Growth (%)')
axes[0].set_title('Month-over-Month Revenue Growth %\n(green = growth, red = decline)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45); axes[0].grid(True, alpha=0.3, axis='y')

# Running total
mom['cumulative_revenue'] = mom['revenue'].cumsum() / 1e6
axes[1].plot(mom.month, mom.cumulative_revenue, color='#f5c842', lw=2.5, marker='o', ms=4)
axes[1].fill_between(mom.month, mom.cumulative_revenue, alpha=0.15, color='#f5c842')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Cumulative Revenue (M TRY)')
axes[1].set_title('Cumulative Revenue Over Time', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q4_mom_growth.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Query 5 — At-Risk Customers (Churn Detection via SQL)

### Business question
Which customers have not ordered recently and are at risk of churning?

### SQL concepts used
- `MAX()` with GROUP BY to find each customer's last order date
- `julianday()` to calculate days since last order
- `CASE WHEN` to create risk segments
- CTE for clean multi-step logic

### Business value
This query directly feeds a re-engagement campaign. Customers with 60-90 days since last order get a discount email. Customers over 90 days get a stronger win-back offer.

In [ ]:
q5 = """
WITH customer_activity AS (
    SELECT
        o.customer_id,
        c.segment,
        c.city,
        COUNT(DISTINCT o.order_id)                       AS total_orders,
        ROUND(SUM(oi.quantity * oi.unit_price), 0)       AS total_spent,
        MAX(o.order_date)                                AS last_order_date,
        CAST(julianday('2024-03-31') -
             julianday(MAX(o.order_date)) AS INTEGER)    AS days_since_last_order
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c    ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id, c.segment, c.city
)
SELECT
    customer_id, segment, city,
    total_orders, total_spent,
    last_order_date, days_since_last_order,
    CASE
        WHEN days_since_last_order <= 30  THEN 'Active'
        WHEN days_since_last_order <= 60  THEN 'At Risk — Low'
        WHEN days_since_last_order <= 90  THEN 'At Risk — Medium'
        ELSE                                   'Churned'
    END AS risk_segment
FROM customer_activity
ORDER BY days_since_last_order DESC
"""

risk_df = pd.read_sql(q5, conn)
risk_summary = risk_df.groupby('risk_segment').agg(
    customers=('customer_id','count'),
    avg_spent=('total_spent','mean'),
    total_value=('total_spent','sum')
).round(0).reset_index()

print('Customer Risk Segments:')
print(risk_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Query 5 — At-Risk Customer Detection', fontsize=13, fontweight='bold')

risk_colors = {'Active':'#3de8a0','At Risk — Low':'#f5c842',
                'At Risk — Medium':'#f06090','Churned':'#8b949e'}
seg_order = ['Active','At Risk — Low','At Risk — Medium','Churned']
risk_plot = risk_summary.set_index('risk_segment').reindex(seg_order)

axes[0].bar(risk_plot.index, risk_plot.customers,
             color=[risk_colors[s] for s in risk_plot.index], alpha=0.85)
for i, v in enumerate(risk_plot.customers):
    axes[0].text(i, v + 5, str(int(v)), ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Customers by Risk Segment', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15); axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(risk_plot.index, risk_plot.total_value / 1e6,
             color=[risk_colors[s] for s in risk_plot.index], alpha=0.85)
axes[1].set_ylabel('Total Historical Spend (M TRY)')
axes[1].set_title('Revenue at Risk by Segment\n(value of customers we could lose)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('q5_at_risk.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

churned_value = risk_plot.loc['Churned','total_value'] if 'Churned' in risk_plot.index else 0
print(f'\nRevenue at risk from churned customers: {churned_value/1e6:.1f}M TRY')
print('Recommendation: target At Risk segments with re-engagement campaigns immediately.')

## Query 6 — Repeat Purchase Rate by City

### Business question
What percentage of customers place more than one order? Does this differ by city?

### SQL concepts used
- Nested aggregation with CASE WHEN
- Percentage calculation inside SQL
- GROUP BY with multiple columns
- HAVING to filter groups with enough customers

### Why this matters
Repeat purchase rate is one of the strongest indicators of customer satisfaction and product-market fit. A city with a low repeat rate might indicate delivery problems, local competition, or a pricing mismatch.

In [ ]:
q6 = """
WITH customer_orders AS (
    SELECT
        o.customer_id,
        c.city,
        c.segment,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id, c.city, c.segment
)
SELECT
    city,
    COUNT(*)                                                   AS total_customers,
    SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END)         AS repeat_customers,
    ROUND(
        100.0 * SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                          AS repeat_rate_pct,
    ROUND(AVG(order_count), 2)                                 AS avg_orders_per_customer
FROM customer_orders
GROUP BY city
HAVING total_customers >= 50
ORDER BY repeat_rate_pct DESC
"""

repeat_df = pd.read_sql(q6, conn)
print('Repeat Purchase Rate by City:')
print(repeat_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Query 6 — Repeat Purchase Rate by City', fontsize=13, fontweight='bold')

axes[0].barh(repeat_df.city, repeat_df.repeat_rate_pct,
              color='#5b8ef0', alpha=0.85)
axes[0].set_xlabel('Repeat Purchase Rate (%)')
axes[0].set_title('% of Customers with 2+ Orders by City', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(repeat_df.repeat_rate_pct):
    axes[0].text(v + 0.3, i, f'{v}%', va='center', fontsize=10)

axes[1].barh(repeat_df.city, repeat_df.avg_orders_per_customer,
              color='#3de8a0', alpha=0.85)
axes[1].set_xlabel('Average Orders per Customer')
axes[1].set_title('Average Orders per Customer by City', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('q6_repeat_rate.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Business Summary

Six SQL queries answered six real business questions:

| Query | Finding | Action |
|---|---|---|
| Monthly revenue | Clear November peak, summer dip | Plan inventory and campaigns around seasonality |
| Category revenue | Electronics highest revenue, Beauty highest margin | Increase Beauty marketing spend |
| Customer LTV | Premium segment drives disproportionate value | Protect Premium customers with loyalty programme |
| MoM growth | November shows strongest growth | Invest more in November campaigns |
| At-risk customers | Significant revenue from churned segment | Launch win-back email campaign immediately |
| Repeat rate | Varies by city — some cities much lower | Investigate delivery quality in low-repeat cities |

### SQL concepts demonstrated
✅ JOINs across multiple tables  
✅ GROUP BY and aggregations  
✅ CTEs (Common Table Expressions)  
✅ Window functions: LAG, ROW_NUMBER  
✅ CASE WHEN for conditional logic  
✅ Date functions  
✅ HAVING for group-level filtering  
✅ Subqueries and nested logic  
✅ Percentage and ratio calculations